# 03 — Data engineering and a pretraining loop

The data pipeline is part of the model. Before optimizing a network, define the task, licensing/consent rules, deduplication policy, provenance, resolution policy, and split unit. Randomly splitting adjacent video frames or samples from the same episode leaks near-duplicates across train and evaluation. Split by a stable group such as source, episode, subject, or time block before sampling.

This is a tiny local manifest example. It does not download data. Image decoding, augmentation, distributed sharding, and robust resume logic are explicit production concerns, not hidden behind a Trainer.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path

manifest = pd.DataFrame([
    {'sample_id': f's{i}', 'image_path': f'data/image_{i}.jpg', 'caption': c, 'group_id': g}
    for i, (c, g) in enumerate([
        ('red block on left', 'episode-a'), ('red block on table', 'episode-a'),
        ('blue cup on right', 'episode-b'), ('blue cup near box', 'episode-b'),
        ('green ball in bin', 'episode-c'), ('green ball beside bin', 'episode-c')])
])
manifest['caption_chars'] = manifest.caption.str.len()
groups = manifest.group_id.drop_duplicates().to_numpy()
rng = np.random.default_rng(21)
rng.shuffle(groups)
test_groups = set(groups[:1])
manifest['split'] = np.where(manifest.group_id.isin(test_groups), 'test', 'train')
print(manifest[['sample_id', 'group_id', 'split', 'caption_chars']])
assert not (set(manifest.query("split == 'train'").group_id) & set(manifest.query("split == 'test'").group_id))

## Manual optimization loop

The loop below is a language-only miniature to expose the mechanics. For a VLM, the batch yields images, token IDs, attention/padding masks, and labels; `model(images, input_ids)` returns token logits and the same next-token loss can be applied. Important details: `zero_grad`, forward, loss, backward, optional gradient clipping, optimizer step, and validation under `no_grad`. A step is not an epoch. Log the number of examples/tokens actually seen.

In [ ]:
torch.manual_seed(5)
vocab_size, width, seq_len = 12, 24, 6
lm = nn.Sequential(nn.Embedding(vocab_size, width), nn.Linear(width, vocab_size))
optimizer = torch.optim.AdamW(lm.parameters(), lr=2e-3, weight_decay=0.01)
train_ids = torch.randint(1, vocab_size, (256, seq_len + 1))
batch_size = 32

for epoch in range(3):
    lm.train()
    order = torch.randperm(len(train_ids))
    total_loss, seen = 0.0, 0
    for indices in order.split(batch_size):
        batch = train_ids[indices]
        inputs, targets = batch[:, :-1], batch[:, 1:]
        optimizer.zero_grad(set_to_none=True)
        logits = lm(inputs)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), targets.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(lm.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += float(loss.detach()) * len(indices)
        seen += len(indices)
    print(f'epoch={epoch + 1} examples={seen} mean_loss={total_loss / seen:.3f}')

## Checkpoints and scale-up checklist

A useful checkpoint includes model, optimizer, scheduler, global step, random-number-generator states, tokenizer/vocabulary version, data-manifest fingerprint, and configuration. Saving weights alone may not reproduce the run. For larger runs, also track tokens per second, data-loader wait, accelerator utilization, peak memory, and validation by task slice. Resume should be tested by interrupting a toy run and comparing the next update.

Before scaling: inspect random samples; deduplicate; check train/validation overlap; record all preprocessing; build a small overfit test; estimate accelerator memory and training time; set a stopping/rollback criterion. Never use internet-scale scraped data without checking rights, privacy, safety, and dataset terms.